# Actor extraction: Stanza NER (training set)

## Setup

In [ ]:
# packages

import warnings
warnings.filterwarnings('ignore')


import pandas as pd
import numpy as np 
import csv
import spacy_stanza

import os
os.getcwd()

In [ ]:
# load nlp stanza
nlp = spacy_stanza.load_pipeline("nl")

In [ ]:
df = pd.read_csv('data/coded_df_actors_full.csv',
                 sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)

# change article_id to integer
df['article_id'] = df['article_id'].astype(int)
print(df.shape)

In [ ]:
df_text = pd.read_csv('data/final_nosarticles.csv',
                      sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)
df_text['page_id'] = df_text['page_id'].astype(int)

df_text = df_text[['page_id', 'Text']].drop_duplicates()

# rename page_id to article_id
df_text.rename(columns = {'page_id': 'article_id'}, inplace = True)

# remove line break
df_text['Text'] = df_text['Text'].str.replace('[LINE_BREAK]', '\n ')

print(df_text.shape)

In [ ]:
# add text to df
df = pd.merge(df, df_text, on = 'article_id', how = 'left')
print(df.shape)

In [ ]:
# drop if actor_type is Geopolitieke entiteit
df = df[df.actor_type != 'Geopolitieke entiteit']

In [ ]:
unique_articles_df = df[['article_id', 'Text']].drop_duplicates()

In [ ]:
nlp = spacy_stanza.load_pipeline("nl")

In [ ]:
def tag_text_stanza(text):
    # Create a Sentence object from the text
    doc = nlp(text)
    
    # Get the tagged spans
    spans = doc.ents

    # drop if entity label is not in ['ORG','PER']
    spans = [span for span in spans if span.label_ in ['ORG', 'PER']]
    
    # Create a list of tuples containing the entity text and label
    entities = [(ent.text, ent.label_) for ent in spans]

    # Create an empty dictionary to store unique combinations
    unique_entities_tags = {}

    # Iterate through the list of named entity tag combinations
    for entity, tag in entities:
        # Check if the combination exists in the dictionary
        if (entity, tag) not in unique_entities_tags:
            # If it doesn't exist, add it to the dictionary
            unique_entities_tags[(entity, tag)] = True

    # Convert the keys of the dictionary back into a list
    unique_entities = list(unique_entities_tags.keys())

    return unique_entities

In [ ]:
# get the first text as test
text = unique_articles_df['Text'].iloc[2]

# tag the text
unique_entities = tag_text_stanza(text)

In [ ]:
unique_articles_df['entities_stanza'] = unique_articles_df['Text'].apply(tag_text_stanza)

In [ ]:
df_exploded = unique_articles_df.explode('entities_stanza')

In [ ]:
# 2. Split the 'entities' tuples into two separate columns: 'entity_name' and 'entity_type'
df_exploded[['entity_name', 'entity_type']] = pd.DataFrame(df_exploded['entities_stanza'].tolist(), index=df_exploded.index)
print(df_exploded.shape)

In [ ]:
# create a function to check if Text contains kabinet in its lower case form
def check_kabinet(text):
    if 'kabinet' in text.lower():
        return True
    else:
        return False
    
# if the text contains kabinet, then add entity_name as 'Het kabinet' and entity_type as 'ORG'
df_exploded['kabinet'] = df_exploded['Text'].apply(check_kabinet)

In [ ]:
# get kabinet into a new df
df_kabinet = df_exploded[df_exploded['kabinet'] == True]
# drop entities_stanza and entity_name and entity_type
df_kabinet = df_kabinet.drop(columns = ['entities_stanza', 'entity_name', 'entity_type'])

df_kabinet['entity_name'] = 'Het kabinet'
df_kabinet['entity_type'] = 'ORG'

In [ ]:
# concat df_exploded and df_kabinet
df_exploded = pd.concat([df_exploded, df_kabinet], axis = 0)
print(df_exploded.shape)

In [ ]:
# drop kabinet column
df_exploded = df_exploded.drop(columns = 'kabinet')

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('stopwords')
stopwords = nltk.corpus.stopwords.words('dutch')
print(stopwords)

# extend the stopwords with their title case form
stopwords_title = [word.title() for word in stopwords]
stopwords.extend(stopwords_title)
print(stopwords)

In [ ]:
# see where the null values are
df_exploded[df_exploded['entity_name'].isnull()]

# drop if entity_name is null
df_exploded = df_exploded.dropna(subset = ['entity_name'])

In [ ]:
import re

# Function to generate name variations (without stopword filtering)
def generate_name_variations(name):
    if not isinstance(name, str):
        return []
    
    # Convert to lowercase and split into parts
    # Keep only alphanumeric words (no symbols or numbers)
    parts = re.findall(r'\b\w+\b', name)
    
    # Remove stopwords from the parts
    parts_filtered = [part for part in parts]
    
    variations = []
    
    if parts_filtered:
        # Use the entire filtered name and the individual parts as variations
        variations.append(' '.join(parts_filtered))  # Add filtered parts as a single variation
        variations.extend(parts_filtered)  # Add individual parts as variations

    return variations

# Create a new column for name variations
df_exploded['name_variations'] = df_exploded['entity_name'].apply(lambda x: generate_name_variations(x))

# Drop from name_variations if instance of name_variations matches stopwords
df_exploded['name_variations'] = df_exploded['name_variations'].apply(lambda x: [variation for variation in x if variation not in stopwords])
df_exploded['name_variations'] = df_exploded['name_variations'].apply(lambda x: [variation for variation in x if re.match(r'\b\w+\b', variation)])
df_exploded['name_variations'] = df_exploded['name_variations'].apply(lambda x: [variation for variation in x if len(variation) > 1])
# drop if variation is number
df_exploded['name_variations'] = df_exploded['name_variations'].apply(lambda x: [variation for variation in x if not variation.isnumeric()])
# keep only unique variations
df_exploded['name_variations'] = df_exploded['name_variations'].apply(lambda x: list(set(x)))

In [ ]:
# see where entity name has punctuation
df_exploded[df_exploded['entity_name'].str.contains(r'[^\w\s]')]

In [ ]:
# Helper function to match whole words
def is_full_word_match(variation, name):
   
    # Use word boundaries to ensure full word matching
    return bool(re.search(r'\b' + re.escape(variation) + r'\b', name))

# Function to map names within each article (case-insensitive, full-word match, and entity type check)
def map_names_within_article(article_df):
    name_mapping = {}
    
    # Step 1: Populate the name mapping with the longest canonical names
    for index, row in article_df.iterrows():
        variations = row['name_variations']
        canonical_name = row['entity_name']
        entity_type = row['entity_type']  # Get the entity type

        for variation in variations:            
            # Check both variation and entity_type match
            for mapped_name, (current_canonical, current_type) in name_mapping.items():
                if is_full_word_match(variation, mapped_name) or is_full_word_match(mapped_name, variation):
                    if entity_type == current_type:  # Ensure entity types match
                        if len(canonical_name) > len(current_canonical):
                            name_mapping[mapped_name] = (canonical_name, entity_type)
            else:
                name_mapping[variation] = (canonical_name, entity_type)

    # Step 2: Replace entity names based on the canonical mapping (with full-word and entity-type check)
    article_df['entity_name_new'] = article_df.apply(
        lambda row: next(
            (canonical for variation, (canonical, type_) in name_mapping.items()
             if (is_full_word_match(variation, row['entity_name']) or is_full_word_match(row['entity_name'], variation)) 
             and type_ == row['entity_type']), 
            row['entity_name']
        ), axis=1
    )
    
    return article_df

In [ ]:
df_exploded_test = df_exploded.groupby(['article_id', 'entity_type'], group_keys=False).apply(map_names_within_article)

In [ ]:
df_exploded_test.shape

In [ ]:
# see where entity_name and entity_name_new are different
differences = df_exploded_test[df_exploded_test['entity_name'] != df_exploded_test['entity_name_new']]
differences.shape

In [ ]:
# see in differences the entity_type ORG
differences[differences['entity_type'] == 'ORG'][['entity_name', 'entity_name_new']].values

In [ ]:
differences[differences['entity_type'] == 'PER'][['entity_name', 'entity_name_new']].values

In [ ]:
# if entity_name is Rutte, then entity_name_new should be changed to Mark Rutte
df_exploded_test['entity_name_new'] = np.where(df_exploded_test['entity_name'].str.strip() == 'Rutte', 'Mark Rutte', df_exploded_test['entity_name_new'])

# if entity_name is Rutte IV or Rutte V, then entity_name_new should be the same
df_exploded_test['entity_name_new'] = np.where(df_exploded_test['entity_name'].str.strip() == 'Rutte IV', 'Rutte IV', df_exploded_test['entity_name_new'])
df_exploded_test['entity_name_new'] = np.where(df_exploded_test['entity_name'].str.strip() == 'Rutte IV.', 'Rutte IV', df_exploded_test['entity_name_new'])
df_exploded_test['entity_name_new'] = np.where(df_exploded_test['entity_name'].str.strip() == 'Rutte V', 'Rutte V', df_exploded_test['entity_name_new'])
df_exploded_test['entity_name_new'] = np.where(df_exploded_test['entity_name'].str.strip() == 'Rutte V.', 'Rutte V', df_exploded_test['entity_name_new'])

### Note: mapping names of organizations do not work here, only Kamer and WHO makes sense

In [ ]:
# Separate the data into persons (PER) and others (ORG, etc.)
df_persons = df_exploded[df_exploded['entity_type'] == 'PER']
df_others = df_exploded[df_exploded['entity_type'] != 'PER']

# Apply the mapping only to persons
df_persons_updated = df_persons.groupby('article_id', group_keys=False).apply(map_names_within_article)

In [ ]:
# in df_others if the entity_name is Kamer then change it to Tweede Kamer or if the entity_name is WHO then change it to Wereldgezondheidsorganisatie WHO
df_others['entity_name_new'] = df_others['entity_name'].apply(lambda x: 'Tweede Kamer' if x == 'Kamer' else 'Wereldgezondheidsorganisatie WHO' if x == 'WHO' else x)

In [ ]:
df_others['entity_name_new'] = df_others['entity_name'].apply(lambda x: 'Tweede Kamer' if x == 'Kamer' else 'Forum voor Democratie' if x == 'Forum' else x)

In [ ]:
# Combine the processed persons back with the rest of the data
df_final = pd.concat([df_persons_updated, df_others]).sort_index()
df_final.shape

In [ ]:
print(df_final.shape)

In [ ]:
duplicated = df_final[df_final.duplicated(subset = ['article_id', 'entity_name_new', 'entity_type'], keep = False)]
print(duplicated.shape)

In [ ]:
# drop duplicates keep with the longest name_variations
df_final = df_final.sort_values('name_variations', key = lambda x: x.str.len(), ascending = False).drop_duplicates(subset = ['article_id',  'entity_name_new', 'entity_type'], keep = 'first')
print(df_final.shape)

In [ ]:
df_final[df_final['article_id'] == 2331140]

In [ ]:
df_final.groupby(['article_id', 'entity_name_new'])['entity_type'].nunique().value_counts()

In [ ]:
# see the entities with more than one type

entities_with_more_than_one_type = df_final.groupby(['article_id', 'entity_name_new'])['entity_type'].nunique().reset_index()
entities_with_more_than_one_type = entities_with_more_than_one_type[entities_with_more_than_one_type['entity_type'] > 1]
# match artoicle text with article id
entities_with_more_than_one_type = entities_with_more_than_one_type.merge(df_final[['article_id']].drop_duplicates(), on = 'article_id')

In [ ]:
# see in df where entity name is Timen
df_final[df_final['entity_name_new'] == 'De Klister'].Text.values

In [ ]:
entities_with_more_than_one_type.entity_name_new.unique()

In [ ]:
# get the df_final rows where indez is in entities with more than one type
df_entities_more_than_one_type = df_final[df_final.index.isin(entities_with_more_than_one_type.index)]
print(df_entities_more_than_one_type.shape)
# remove the indeces from df_final
df_final = df_final[~df_final.index.isin(entities_with_more_than_one_type.index)]
print(df_final.shape)

In [ ]:
people_names = ['Van Ark', 'Timen', 'Grapperhaus', 'Tedros', 'Ollongren', 'Sake de Vlas', 'Magufuli', 'Breun', 'Trump', 'Helder']
org_names = ['Evofenedex', 'Paaspop', 'Roche', 'Philips', 'Inzek',
       'Trouw', 'Tönnies', 'Abu Dhabi', 'Saltro',
       'Sanquin', 'Tuskegee', 'Wbbbg', 'QALY', 'AstraZeneca',
       'Pointer', 'De Klister', 'Thús', 'Johnsons', 'Jumbo-Visma']

In [ ]:
# in df_entities_more_than_one_type if entity_name_new is in people_names then entity_type should be PER
df_entities_more_than_one_type['entity_type'] = np.where(df_entities_more_than_one_type['entity_name_new'].isin(people_names), 'PER', df_entities_more_than_one_type['entity_type'])
# in df_entities_more_than_one_type if entity_name_new is in org_names then entity_type should be ORG
df_entities_more_than_one_type['entity_type'] = np.where(df_entities_more_than_one_type['entity_name_new'].isin(org_names), 'ORG', df_entities_more_than_one_type['entity_type'])

In [ ]:
# combine df_final and df_entities_more_than_one_type
df_updated = pd.concat([df_final, df_entities_more_than_one_type]).sort_index()
print(df_updated.shape)

In [ ]:
# see if there are any duplicates
duplicated = df_updated[df_updated.duplicated(subset = ['article_id', 'entity_name', 'entity_type'], keep = False)]

In [ ]:
# keep first
df_updated = df_updated.drop_duplicates(subset = ['article_id', 'entity_name', 'entity_type'], keep = 'first')
print(df_updated.shape)

In [ ]:
# update df_final to df_updated
df_final = df_updated
print(df_final.shape)

In [ ]:
# drop entity_name and change entity_name_new to entity_name
df_final = df_final.drop(columns = ['entity_name'])
df_final.rename(columns = {'entity_name_new': 'entity_name'}, inplace = True)

In [ ]:
print(df_final.shape)

In [ ]:
# Define a function to get the length of the name_variations list
df_final['name_variations_length'] = df_final['name_variations'].apply(len)

# Sort the DataFrame by article_id, entity_name, and the length of name_variations in descending order
df_sorted = df_final.sort_values(by=['article_id', 'entity_name', 'name_variations_length'], ascending=[True, True, False])

# Drop duplicates by keeping the first occurrence, which will be the one with the longest name_variations
df_unique = df_sorted.drop_duplicates(subset=['article_id', 'entity_name'], keep='first')

# Drop the helper column
df_unique = df_unique.drop(columns=['name_variations_length'])

print(df_unique.shape)

In [ ]:
# remove special characters or punctuation from entity_name
# see if entity_name contains special characters or punctuation
df_unique[df_unique['entity_name'].str.contains(r'[^\w\s]')].sample(50, random_state = 0)

In [ ]:
# Define Dutch stopwords related to names (adjust as needed)
name_stopwords = {"van", "den", "de", "het", "der", "te", "ten", "ter"}

def generate_name_variations(name):
    if not isinstance(name, str):
        return []
    
    # Extract words while keeping full name intact
    parts = name.split()  # Split by spaces to maintain structure
    
    # Remove stopwords **only** for individual parts, not full names
    filtered_parts = [part for part in parts if part.lower() not in name_stopwords]
    
    variations = set()
    
    if filtered_parts:
        variations.add(" ".join(filtered_parts))  # Full name without stopwords
        variations.update(filtered_parts)  # Individual words (excluding stopwords)
    
    variations.add(name)  # Always include full original name
    
    # Ensure all variations are unique and valid
    variations = {var for var in variations if len(var) > 1 and not var.isnumeric()}
    
    return list(variations)

# Create a new column for name variations
df_unique['name_variations'] = df_unique['entity_name'].apply(lambda x: generate_name_variations(x))

# Drop from name_variations if instance of name_variations matches stopwords
df_unique['name_variations'] = df_unique['name_variations'].apply(lambda x: [variation for variation in x if variation not in stopwords])
df_unique['name_variations'] = df_unique['name_variations'].apply(lambda x: [variation for variation in x if re.match(r'\b\w+\b', variation)])
df_unique['name_variations'] = df_unique['name_variations'].apply(lambda x: [variation for variation in x if len(variation) > 1])
df_unique['name_variations'] = df_unique['name_variations'] + df_unique['entity_name'].apply(lambda x: [x])

# drop if variation is number
df_unique['name_variations'] = df_unique['name_variations'].apply(lambda x: [variation for variation in x if not variation.isnumeric()])
# keep only unique variations
df_unique['name_variations'] = df_unique['name_variations'].apply(lambda x: list(set(x)))

In [ ]:
# sort the df based on article_id, entity_name, and entity_type

df_unique = df_unique.sort_values(['article_id', 'entity_name', 'entity_type'])

In [ ]:
counts_entitytypes = df_unique.groupby(['article_id','entity_name'])['entity_type'].nunique().reset_index()

In [ ]:
# see where entity_name includes McDonald

df_unique[df_unique['entity_name'] == 'D.']

In [ ]:
# see where entity name is Bas van den Putte
df_unique[df_unique['entity_name'] == 'Bas van den Putte']

In [ ]:
import re
import stanza

# Load Dutch Stanza model
nlp = stanza.Pipeline("nl", processors="tokenize")

In [ ]:
def extract_sentences_stanza(text, name_variations):
    if not isinstance(text, str) or not text.strip():
        return []  # Return empty list if text is missing
    
    # Use Stanza for sentence splitting
    doc = nlp(text)
    sentences = [sentence.text for sentence in doc.sentences]

    # Ensure exact match with correct case
    relevant_sentences = [
        sentence for sentence in sentences
        if any(re.search(rf'\b{name}\b', sentence) for name in name_variations)  # Exact match, case-sensitive
    ]

    return relevant_sentences

# Apply function to dataframe
df_unique['relevant_sentences'] = df_unique.apply(lambda x: extract_sentences_stanza(x['Text'], x['name_variations']), axis=1)

In [ ]:
df_unique[df_unique['entity_name'] == 'Bikker'].relevant_sentences.values

In [ ]:
# make exploded_sentences a string by joining the list of sentences
df_unique['relevant_sentences_string'] = df_unique['relevant_sentences'].apply(lambda x: ' \n'.join(x))
df_unique.relevant_sentences.values[0]

In [ ]:
df_unique[df_unique['entity_name'] == 'Het kabinet']['relevant_sentences_string'].values[0]

In [ ]:
# save the df to csv 
df_unique.to_csv('data/actor_entities_extracted_v3.csv', index = False, sep=';', quoting=csv.QUOTE_NONNUMERIC, encoding = 'utf-8')

# Create the actor training df

In [ ]:
# limit to only articles about covid
df_covid = df[df['about_covid'] == 1]

In [ ]:
# create variable measures by taking the max of all measure_ variables
df_covid['measures'] = df_covid[['measure_1', 'measure_2', 'measure_3', 'measure_4', 'measure_5',
                        'measure_6', 'measure_7', 'measure_8', 'measure_9', 'measure_10',
                        'measure_11', 'measure_12', 'measure_13', 'measure_14', 'measure_15',
                        'measure_16', 'measure_17']].max(axis = 1)

In [ ]:
# create variable positive measures by taking the max of all measure_ variables
df_covid['measures_positive'] = df_covid[['measure_1_positive', 'measure_2_positive', 'measure_3_positive', 'measure_4_positive', 'measure_5_positive',
                              'measure_6_positive', 'measure_7_positive', 'measure_8_positive', 'measure_9_positive', 'measure_10_positive',
                                'measure_11_positive', 'measure_12_positive', 'measure_13_positive', 'measure_14_positive', 'measure_15_positive',
                                'measure_16_positive', 'measure_17_positive']].max(axis = 1)

In [ ]:
# create variable negative measures by taking the max of all measure_ variables
df_covid['measures_negative'] = df_covid[['measure_1_negative', 'measure_2_negative', 'measure_3_negative', 'measure_4_negative', 'measure_5_negative',
                              'measure_6_negative', 'measure_7_negative', 'measure_8_negative', 'measure_9_negative', 'measure_10_negative',
                                'measure_11_negative', 'measure_12_negative', 'measure_13_negative', 'measure_14_negative', 'measure_15_negative',
                                'measure_16_negative', 'measure_17_negative']].max(axis = 1)

In [ ]:
# create variable neutral measures by taking the max of all measure_ variables
df_covid['measures_neutral'] = df_covid[['measure_1_neutral', 'measure_2_neutral', 'measure_3_neutral', 'measure_4_neutral', 'measure_5_neutral',
                              'measure_6_neutral', 'measure_7_neutral', 'measure_8_neutral', 'measure_9_neutral', 'measure_10_neutral',
                                'measure_11_neutral', 'measure_12_neutral', 'measure_13_neutral', 'measure_14_neutral', 'measure_15_neutral',
                                'measure_16_neutral', 'measure_17_neutral']].max(axis = 1)

In [ ]:
df_selected = df_covid[df_covid['coder'] == 'main_coder'][['article_id', 'coder', 'actor_name', 'actor_type', 'directly_quoted', 'indirectly_quoted', 'actor_function', 'actor_pp', 'talks_covid_measures', 'measures','measures_positive', 'measures_negative', 'measures_neutral']].drop_duplicates()
df_selected['actor_name_normalized'] = df_selected['actor_name'].str.lower()
# drop if actor name is nan
df_selected = df_selected.dropna(subset = ['actor_name_normalized'])

In [ ]:
# get name_variations for each actor_name
# Create a new column for name variations
df_selected['name_variations'] = df_selected['actor_name'].apply(lambda x: generate_name_variations(x))

# Drop from name_variations if instance of name_variations matches stopwords
df_selected['name_variations'] = df_selected['name_variations'].apply(lambda x: [variation for variation in x if variation not in stopwords])
df_selected['name_variations'] = df_selected['name_variations'].apply(lambda x: [variation for variation in x if re.match(r'\b\w+\b', variation)])
df_selected['name_variations'] = df_selected['name_variations'].apply(lambda x: [variation for variation in x if len(variation) > 1])
df_selected['name_variations'] = df_selected['name_variations'] + df_selected['actor_name'].apply(lambda x: [x])

# drop if variation is number
df_selected['name_variations'] = df_selected['name_variations'].apply(lambda x: [variation for variation in x if not variation.isnumeric()])
# keep only unique variations
df_selected['name_variations'] = df_selected['name_variations'].apply(lambda x: list(set(x)))

In [ ]:
# see where actor name lower version is de kamer
df_selected[df_selected['actor_name'].str.lower() == 'de kamer']
# change where lower version is de kamer to Tweede Kamer
df_selected['actor_name'] = np.where(df_selected['actor_name'].str.lower() == 'de kamer', 'Tweede Kamer', df_selected['actor_name'])
df_selected['actor_name_normalized'] = np.where(df_selected['actor_name_normalized'].str.lower() == 'de kamer', 'tweede kamer', df_selected['actor_name_normalized'])

In [ ]:
df_selected['actor_name'] = np.where(df_selected['actor_name'].str.lower() == 'de inspectie', 'Inspectie Gezondheidszorg en Jeugd', df_selected['actor_name'])
df_selected['actor_name_normalized'] = np.where(df_selected['actor_name_normalized'].str.lower() == 'de inspectie', 'inspectie gezondheidszorg en jeugd', df_selected['actor_name_normalized'])

In [ ]:
df_selected[df_selected['actor_name'].str.lower() == 'de kamer']

In [ ]:
# any duplicates?
duplicated = df_selected[df_selected.duplicated(subset = ['article_id', 'actor_name', 'actor_type'], keep = False)]

In [ ]:
df_unique_selected = df_unique[['article_id', 'entity_name', 'entity_type', 'relevant_sentences_string', 'name_variations']]
df_unique_selected['entity_name_normalized'] = df_unique_selected['entity_name'].str.lower()
# drop name variations
df_unique_selected.drop(columns = ['name_variations'], inplace = True)

In [ ]:
# any duplicates?
duplicated = df_unique_selected[df_unique_selected.duplicated(subset = ['article_id', 'entity_name', 'entity_type'], keep = False)]

In [ ]:
# if entity_type is PER then it is Persoon, if ORG then it is Organisatie
df_unique_selected['entity_type'] = np.where(df_unique_selected['entity_type'] == 'PER', 'Persoon', 'Organisatie')

In [ ]:
# in df_selected change actor_type to entity_type
df_selected.rename(columns = {'actor_type': 'entity_type'}, inplace = True)

In [ ]:
from fuzzywuzzy import fuzz

# Step 1: Merge on both article_id and entity_type
merged_df = pd.merge(df_selected, df_unique_selected, on=['article_id', 'entity_type'], how='inner')

# Step 2: Calculate string similarity and keep matches
def match_names(row):
    actor_name = row['actor_name_normalized']
    entity_name = row['entity_name_normalized']
    return fuzz.token_set_ratio(actor_name, entity_name)

# Apply the matching function
merged_df['similarity'] = merged_df.apply(match_names, axis=1)

# Step 3: Filter based on threshold
threshold = 90
final_matches = merged_df[merged_df['similarity'] >= threshold]

In [ ]:
# check the matches below 90 similarity
final_matches[final_matches['similarity'] == 100][['actor_name', 'entity_name']].drop_duplicates()

In [ ]:
print(final_matches.shape)
print(df_selected.shape)
print(df_unique_selected.shape)    

In [ ]:
# save final_matches to explore
final_matches.to_excel('data/final_matches_v3.xlsx', index = False)

In [ ]:
# get the actors who are not in the final_matches based on article_id and actor_name
len(df_selected[~df_selected['actor_name'].isin(final_matches['actor_name'])])

In [ ]:
1 - (356/len(df_selected))

In [ ]:
notmatched = df_selected[~df_selected['actor_name'].isin(final_matches['actor_name'])]

In [ ]:
df_final[df_final['article_id'] == 2334206]

In [ ]:
# make a column quoted == 1 for final_matches
final_matches['quoted'] = 1

In [ ]:
# select the df that will be used for the model

model_df = final_matches[['article_id', 'entity_name', 'entity_type', 'relevant_sentences_string', 'quoted', 'directly_quoted', 'indirectly_quoted', 'actor_function','actor_pp',
                          'talks_covid_measures', 'measures', 'measures_positive', 'measures_negative', 'measures_neutral']]

In [ ]:
# get the entities that are not in final_matches
df_unique_selected[~df_unique_selected['entity_name'].isin(final_matches['entity_name'])].head()

df_not_quoted = df_unique_selected[~df_unique_selected['entity_name'].isin(final_matches['entity_name'])]

In [ ]:
df_not_quoted['quoted'] = 0
df_not_quoted['directly_quoted'] = 0
df_not_quoted['indirectly_quoted'] = 0


# select the relevant columns to match with model_df
df_not_quoted = df_not_quoted[['article_id', 'entity_name', 'entity_type', 'relevant_sentences_string', 'quoted', 'directly_quoted', 'indirectly_quoted']]

In [ ]:
df_not_quoted.shape

In [ ]:
# combine model_df and df_not_quoted
model_df = pd.concat([model_df, df_not_quoted])
print(model_df.shape)

In [ ]:
# create input text by combining entity_name : and relevant_sentences_string
model_df['input_text'] = model_df['entity_name'] + ':\n ' + model_df['relevant_sentences_string']

In [ ]:
# see where input_text is null
model_df[model_df['input_text'].isnull()]

In [ ]:
# if input text is null drop the row
model_df = model_df.dropna(subset = ['input_text'])

In [ ]:
# save the df
model_df.to_csv('data/actors_training_df_v3.csv', index = False, sep=';', quoting=csv.QUOTE_NONNUMERIC, encoding = 'utf-8')